# Benchmark Speech to Text (ST) Models 

In [2]:
# Standard library
import time 
import re
from pathlib import Path

# Core
import numpy as np
import pandas as pd

# Audio
import librosa
import soundfile as sf

# ML / ASR
import torch
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    AutoProcessor,
    AutoModelForCTC,
    Wav2Vec2ForCTC
)

# Utils
from tqdm.auto import tqdm
from jiwer import wer, cer

/Users/dmatekenya/git-repos/chichewa-asr/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Set up and Configs 

In [3]:
# =============================
# BASE WORKING DIRS
# ============================
PLATFORM = 'Local-Mac' # 'Lambda-Jupyter-Hub'

if PLATFORM == 'Lambda-Jupyter-Hub':
    DIR_TEST = Path('/home/ubuntu/audios-test')
else:
    # Speech data folder in Google Drive
    DIR_DATA_GOOGLE_DRIVE = Path("/Users/dmatekenya/Library/CloudStorage/GoogleDrive-dmatekenya@gmail.com/My Drive/Chichewa-ASR2/data/chichewa-speech-data/")
    
    DIR_DATA = Path().cwd().parents[0]/"data"
    DIR_TEST = DIR_DATA / "test"
    DIR_DEV = DIR_DATA / "dev"

FILE_TEST_METADATA = DIR_TEST / "metadata.csv"
FILE_BENCHMARK_RESULTS = DIR_DATA / "model-benchmarks/results-whisper-mms.csv"

# =============================
# API KEYS
# =============================
# load_dotenv()
# INTRON_API_KEY = os.getenv("INTRON_API_KEY")

# =============================
# GPU Device CONFIG
# =============================
TARGET_SR = 16000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 2.Evaluate Open Source Models
- Whisper family
- MMS family
- Conformer
- SpeechLLMs

### 2.1. Whisper Models 

In [6]:
def load_audio_16k(audio_path, target_sr=16000):
    audio, sr = librosa.load(audio_path, sr=target_sr, mono=True)
    return audio

In [7]:
def load_whisper_model(model_id, language="Shona", task="transcribe"):
    processor = WhisperProcessor.from_pretrained(
        model_id,
        language=language,
        task=task,
    )
    
    model = WhisperForConditionalGeneration.from_pretrained(model_id)
    model = model.to(DEVICE)
    
    if DEVICE == "cuda":
        model = model.to(torch.float16)
    
    model.eval()
    
    return processor, model

In [8]:
def transcribe_whisper(processor, model, audio_16k, target_sr=16000):
    feats = processor(
        audio_16k,
        return_tensors="pt",
        sampling_rate=target_sr
    ).input_features

    model_device = next(model.parameters()).device
    model_dtype = next(model.parameters()).dtype

    feats = feats.to(device=model_device, dtype=model_dtype)

    with torch.no_grad():
        generated_ids = model.generate(input_features=feats)

    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return text.strip()

In [9]:
def transcribe_whisper_forced_decoder(processor, model, audio_16k, target_sr=16000):
    feats = processor(
        audio_16k,
        return_tensors="pt",
        sampling_rate=target_sr
    ).input_features

    model_device = next(model.parameters()).device
    model_dtype = next(model.parameters()).dtype

    feats = feats.to(device=model_device, dtype=model_dtype)

    forced_decoder_ids = processor.get_decoder_prompt_ids(
        language="shona",
        task="transcribe"
    )

    with torch.no_grad():
        generated_ids = model.generate(
            input_features=feats,
            forced_decoder_ids=forced_decoder_ids
        )

    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return text.strip()

In [22]:
def evaluate_whisper(
    df_metadata,
    whisper_models,
    dir_test,
    target_sr=16000,
    num_samples=None,
    df_results=None,
):
    import time
    from pathlib import Path

    # subset for debugging
    df = df_metadata.head(num_samples).copy() if num_samples is not None else df_metadata.copy()

    # existing completed inferences
    if df_results is not None and len(df_results) > 0:
        completed = set(
            zip(
                df_results["model"].astype(str),
                df_results["audio_file"].astype(str),
            )
        )
        results = df_results.to_dict("records")
    else:
        completed = set()
        results = []

    for model_name, model_id in whisper_models.items():
        print(f"\nRunning: {model_name} | candidate samples={len(df)}")

        # only keep rows not yet done for this model
        df_model = df[
            ~df["audio_filename"].apply(
                lambda x: (str(model_name), Path(x).name) in completed
            )
        ].copy()

        print(f"Skipping {len(df) - len(df_model)} already completed")
        print(f"Remaining for {model_name}: {len(df_model)}")

        if len(df_model) == 0:
            continue

        processor, model = load_whisper_model(model_id)

        for row in df_model.itertuples():
            audio_path = Path(dir_test) / row.audio_filename
            audio_file = audio_path.name

            # extra safeguard
            if (str(model_name), audio_file) in completed:
                continue

            audio_16k = load_audio_16k(audio_path, target_sr=target_sr)

            start_time = time.time()
            pred_text = transcribe_whisper_forced_decoder(
                processor, model, audio_16k, target_sr=target_sr
            )
            inference_time = time.time() - start_time

            ref_text = row.transcript
            wer_score = wer(ref_text, pred_text)

            record = {
                "model": model_name,
                "audio_file": audio_file,
                "ref_text": ref_text,
                "pred_text": pred_text,
                "wer": round(wer_score * 100, 2),
                "inference_time_sec": inference_time,
            }

            results.append(record)
            completed.add((str(model_name), audio_file))

        del model
        del processor
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.DataFrame(results)

In [23]:
WHISPER_MODELS = {
    "whisper-small": "openai/whisper-small",
    "whisper-medium": "openai/whisper-medium",
    "whisper-large-v2": "openai/whisper-large-v2",
    'whisper-large-v3': "openai/whisper-large-v3",
    'whisper-large-v3-turbo': "openai/whisper-large-v3-turbo"}

In [121]:
# =================================================
# LOAD METADATA AND PREVIOUS WHISPER RESULTS
# =================================================

df_metadata = pd.read_csv(FILE_TEST_METADATA)

# Load previous results and rename model to include "whisper" suffix
df_results = pd.read_csv('results_whisper_small_mediaum.csv')
df_results = df_results[['model', 'audio_file', 'ref_text', 'pred_text', 'wer', "inference_time_sec"]]
df_results['model'] = df_results.model.apply(lambda x : 'whisper-{}'.format(x))

In [36]:
# =================================================
# EVALUATE WLL WHISPER MODELS 
# =================================================
start = time.time()
df_seq2 = evaluate_whisper(
    df_metadata,
    WHISPER_MODELS,
    DIR_TEST,
    df_results= df_results
)
seq_runtime = time.time() - start

print(f"Sequential total runtime: {seq_runtime:.2f} sec")


Running: whisper-small | candidate samples=573
Skipping 573 already completed
Remaining for whisper-small: 0

Running: whisper-medium | candidate samples=573
Skipping 470 already completed
Remaining for whisper-medium: 103


Loading weights: 100%|██████████| 947/947 [00:00<00:00, 8466.93it/s]



Running: whisper-large-v2 | candidate samples=573
Skipping 0 already completed
Remaining for whisper-large-v2: 573


Loading weights: 100%|██████████| 1259/1259 [00:00<00:00, 8458.44it/s]
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!



Running: whisper-large-v3 | candidate samples=573
Skipping 0 already completed
Remaining for whisper-large-v3: 573


Loading weights: 100%|██████████| 1259/1259 [00:00<00:00, 8461.38it/s]
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!



Running: whisper-large-v3-turbo | candidate samples=573
Skipping 0 already completed
Remaining for whisper-large-v3-turbo: 573


Loading weights: 100%|██████████| 587/587 [00:00<00:00, 8217.51it/s]
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!


Sequential total runtime: 3785.72 sec


In [87]:
# ======================================================
# COMBINE ALL WHISPER RESULTS
# ======================================================
df_whisper.drop_duplicates(subset=['model', 'audio_file'], inplace=True)
df_whisper = pd.concat([df_results, df_seq2], axis=0)

# Add GPU to give an idea of the platform the benchmarking was run on
df_whisper['gpu'] = 'GH200-96gb'
df_whisper.to_csv('results-whisper.csv', index=False)

### 2.2. Wave2Vec Model Family

In [52]:
# =======================================
# Declare MMS Models to try
# =======================================

MMS_MODELS = {
    "mms-1b-all": "facebook/mms-1b-all",
    'mms-1b-l1107': "facebook/mms-1b-l1107",
    'mms-1b-l1102': "facebook/mms-1b-fl102"
}

In [53]:
def load_mms_model(model_id: str, target_lang='nya'):
    processor = AutoProcessor.from_pretrained(
        model_id,
        target_lang=target_lang,
    )
    model = Wav2Vec2ForCTC.from_pretrained(
        model_id,
        target_lang=target_lang,
        ignore_mismatched_sizes=True,
    ).to(DEVICE)
    model.eval()
    return processor, model

In [54]:
def transcribe_mms(processor, model, audio_16k, target_sr=16000):
    inputs = processor(
        audio_16k,
        sampling_rate=target_sr,
        return_tensors="pt",
        padding=True
    )

    inputs = {k: v.to(next(model.parameters()).device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits

    pred_ids = torch.argmax(logits, dim=-1)
    text = processor.batch_decode(pred_ids)[0]
    return text.strip()

In [55]:
def evaluate_mms_models(
    df_audio_metadata,
    audio_samples=None,
    dir_test=DIR_TEST,
    target_sr=TARGET_SR,
):
    results = []

    # Optional subset for debugging
    if audio_samples is not None:
        df_audio_metadata = df_audio_metadata.head(audio_samples).copy()
    else:
        df_audio_metadata = df_audio_metadata.copy()

    for model_name, model_id in MMS_MODELS.items():
        print(f"\nRunning MMS model: {model_name} | samples={len(df_audio_metadata)}")

        processor, model = load_mms_model(model_id)

        for row in df_audio_metadata.itertuples():
            audio_path = Path(dir_test) / row.audio_filename
            audio_16k = load_audio_16k(audio_path, target_sr=target_sr)

            start_time = time.time()
            pred_text = transcribe_mms(
                processor,
                model,
                audio_16k,
                target_sr=target_sr
            )
            inference_time = time.time() - start_time

            ref_text = row.transcript
            wer_score = wer(ref_text, pred_text)

            results.append({
                "model": model_name,
                "audio_file": audio_path.name,
                "ref_text": ref_text,
                "pred_text": pred_text,
                "wer": round(wer_score * 100, 2),
                "inference_time_sec": inference_time,
            })

        # cleanup
        del model
        del processor
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    results_df = pd.DataFrame(results)
    return results_df

In [56]:
df_metadata = pd.read_csv(FILE_TEST_METADATA)
df_mms = evaluate_mms_models(df_metadata)


Running MMS model: mms-1b-all | samples=573


Loading weights: 100%|██████████| 1096/1096 [00:00<00:00, 10377.16it/s]
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!



Running MMS model: mms-1b-l1107 | samples=573


Loading weights: 100%|██████████| 1096/1096 [00:00<00:00, 8083.51it/s]
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!



Running MMS model: mms-1b-l1102 | samples=573


Loading weights: 100%|██████████| 1096/1096 [00:00<00:00, 8189.10it/s]
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!
[src/libmpg123/layer3.c:INT123_do_layer3():1804] error: dequantization failed!


In [111]:
def add_wer_pct(df, df_raw):
    df = df.copy()

    # default: assume already percentage
    df["wer_pct"] = df["wer"].round(2)

    # create lookup set of raw rows
    raw_pairs = set(zip(df_raw["model"], df_raw["audio_file"]))

    # mask rows that need conversion
    mask_raw = pd.Series(
        [(m, a) in raw_pairs for m, a in zip(df["model"], df["audio_file"])],
        index=df.index
    )

    # convert only those rows
    df.loc[mask_raw, "wer_pct"] = (df.loc[mask_raw, "wer"] * 100).round(2)

    return df

In [124]:
# =======================================================
# COMBINE WHISPER AND MMS RESULTS
# =======================================================
# Dataframe with whisper-small medium with raw WER
df_raw = pd.read_csv("results_whisper_small_mediaum.csv")[["model", "audio_file"]]
df_raw['model'] = df_raw.model.apply(lambda x : 'whisper-{}'.format(x))

df_whisper_mms = pd.concat([df_mms, df_whisper], axis=0)
df_whisper_mms2 = add_wer_pct(df_whisper_mms, df_raw)

# Save results
df_whisper_mms2['gpu'] = 'GH200-96gb'
df_whisper_mms2.to_csv('results-whisper-mms.csv', index=False)


## 2.3 Other Open Sourcer Models

In [125]:
other_open_source_models = {'Cohere': 'CohereLabs/cohere-transcribe-03-2026',
                            'Qwen3-ASR-1.7B': 'Qwen/Qwen3-ASR-1.7B', 'Qwen3-ASR-0.6B': 'Qwen/Qwen3-ASR-0.6B', 'Mistral': 'mistralai/Voxtral-Mini-4B-Realtime-2602'}

## 3. Analyse Results

In [4]:
df_benchmark = pd.read_csv(FILE_BENCHMARK_RESULTS)
df_metadata = pd.read_csv(FILE_TEST_METADATA)


In [ ]:
df_merged = df_metadata.merge(df_benchmark, left_on="audio_filename", right_on="audio_file", suffixes=("_meta", "_bench"))
df_merged.head()

In [5]:
summary = (
    df_benchmark.groupby("model")
    .agg(
        mean_wer=("wer_pct", "mean"),
        median_wer=("wer_pct", "median"),
        p90_wer=("wer_pct", lambda x: x.quantile(0.9)),
        capped_mean_wer=("wer_pct", lambda x: x.clip(upper=100).mean()),
    )
    .round(2)
    .sort_values("mean_wer")
)

summary

,mean_wer,median_wer,p90_wer,capped_mean_wer
model,,,,
mms-1b-all,78.00,80.00,100.00,76.87
mms-1b-l1107,81.12,83.33,100.00,80.77
mms-1b-l1102,81.78,80.43,100.00,77.13
whisper-large-v3,127.78,106.80,156.18,98.35
whisper-large-v2,186.16,100.00,226.61,99.01
whisper-large-v3-turbo,193.50,105.13,379.95,98.42
whisper-small,236.20,100.00,584.21,99.73
whisper-medium,244.93,100.00,389.57,99.00
